# 독서 토론 코칭 시스템 — 평가 노트북 (LLM-as-a-Judge + 사람 보정)

## 이 노트북이 보여주는 것 (피드백 반영 v0.2)
1. **30+ 골든셋**으로 3개 코칭 스타일(v1/v2/v3)의 *후속 질문 품질*을 4축 루브릭으로 채점.
2. **환각/그라운딩 점검** — 책 요약·사용자 발화에 없는 장면을 지어냈는지(`fabrication`) 별도 플래그.
3. **사람 평가와의 상관 검증(Calibration)** — 골든셋 일부를 사람이 같은 4축으로 채점하고
   Judge 점수와 **Spearman 상관**을 계산. 임계값(0.70) 미만이면 Judge를 신뢰하지 않는다.
4. **비용·지연·안전·품질**을 함께 집계하고, 대시보드가 읽을 `eval_summary.json`을 남긴다.

> **학습(파인튜닝/RLHF) 없음.** 본 노트북은 프롬프트 버전 × 골든셋 × 평가의 회귀 루프만 다룬다.

| 축 | 정의 (그라운딩 재정의) | 가중치 |
|----|------|--------|
| Openness | Yes/No로 닫히지 않고 사고를 확장시키는가 | 0.2 |
| Groundedness | **사용자 발화 또는 제공된 책 메타데이터**에 닻을 내렸는가 (지어낸 인용 아님) | 0.3 |
| Specificity | 추상적 일반론이 아니라, 사용자의 말/메타에서 끌어온 구체성이 있는가 | 0.2 |
| Provocativeness | 사용자가 다시 답하고 싶을 만큼 흥미를 끄는가 | 0.3 |

Question Depth Score = 가중 평균 (1~5)


In [ ]:
import os, json, time, math, random
import pandas as pd
import yaml

# 설정값 로드 (임계값/골든셋 규모를 코드에 하드코딩하지 않음)
with open('../configs/model.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)
JUDGE_MODEL = CFG['judge']['model_name']
MIN_CORR = CFG['judge']['min_human_correlation']        # 0.70
MIN_CASES = CFG['judge']['calibration_min_cases']       # 30

API_URL = os.getenv('API_URL', 'http://localhost:8000')

# ── 오프라인 모드 ──────────────────────────────────────────────
# API 키/백엔드가 없으면 모의(mock) 데이터로 '파이프라인 전체'를 재현한다.
# (채점 로직·상관 계산·집계·시각화·산출물 저장까지 그대로 검증 가능)
USE_API = bool(os.getenv('OPENAI_API_KEY'))
try:
    import requests
    if USE_API:
        requests.get(f'{API_URL}/health', timeout=2)
except Exception:
    USE_API = False

random.seed(7)
print(f'JUDGE_MODEL={JUDGE_MODEL} | MIN_CORR={MIN_CORR} | MIN_CASES={MIN_CASES}')
print('MODE =', 'LIVE API' if USE_API else 'OFFLINE (mock) — 파이프라인 검증용')


## 1. Golden Set — 책 10권 × 시나리오 3종 = 30 케이스

세 시나리오: **인상깊은 구절 공유 / 동의하지 않는 부분 / 내 삶에 적용**.
피드백 권고(30~50개)를 충족한다.

In [ ]:
import json, os
# 골든셋은 data/golden_set.json(DVC 버전관리)을 단일 출처로 로드. 없으면 인라인 폴백.
_gp = "../data/golden_set.json"
if os.path.exists(_gp):
    GOLDEN_SET = json.load(open(_gp, encoding="utf-8"))["cases"]
else:
    GOLDEN_BOOKS = {
        '데미안': {
            '인상깊은 구절 공유': '"새는 알을 깨고 나온다"는 구절이 마음에 박혔어요. 익숙한 세계를 부수는 일 같았어요.',
            '동의하지 않는 부분': '싱클레어가 데미안에게 지나치게 의존하는 게 성장이라기보다 또 다른 종속처럼 느껴졌어요.',
            '내 삶에 적용': '저도 제 안의 안전한 세계를 깨야 할 때가 온 것 같아 두렵습니다.',
        },
        '1984': {
            '인상깊은 구절 공유': '"전쟁은 평화다"라는 표어가 처음엔 말장난 같았는데 점점 무섭게 다가왔어요.',
            '동의하지 않는 부분': '윈스턴이 결국 자유의 환상을 포기하는 결말이 너무 패배주의적이라고 느꼈어요.',
            '내 삶에 적용': '제가 알고리즘이 보여주는 것만 진실로 믿어온 건 아닌지 돌아보게 됐어요.',
        },
        '소년이 온다': {
            '인상깊은 구절 공유': '죽은 자의 목소리로 서술되는 장면에서 한참 책을 덮고 있었어요.',
            '동의하지 않는 부분': '고통을 이렇게까지 직시하게 만드는 게 독자에게 윤리적으로 정당한지 의문이 들었어요.',
            '내 삶에 적용': 'SNS에서 폭력을 가볍게 소비해온 제 모습이 부끄러워졌습니다.',
        },
        '죽음의 수용소에서': {
            '인상깊은 구절 공유': '의미가 있으면 어떤 고통도 견딜 수 있다는 말이 부럽기도 무섭기도 했어요.',
            '동의하지 않는 부분': '의미를 찾는 책임을 개인에게만 지우는 건 가혹하다는 생각이 들었어요.',
            '내 삶에 적용': '제 일에서 \'왜\'를 잃어버린 채 버티고만 있다는 걸 깨달았어요.',
        },
        '이방인': {
            '인상깊은 구절 공유': '"오늘 엄마가 죽었다"라는 첫 문장의 무심함이 오래 남았어요.',
            '동의하지 않는 부분': '뫼르소의 무감각을 정직함으로 포장하는 해석에 동의하기 어려웠어요.',
            '내 삶에 적용': '저도 사회가 기대하는 감정을 연기하며 사는 건 아닌지 생각했어요.',
        },
        '어린 왕자': {
            '인상깊은 구절 공유': '"길들인다는 것은 관계를 맺는 것"이라는 말이 다르게 읽혔어요.',
            '동의하지 않는 부분': '어른을 너무 단순하게 희화화한 게 아쉬웠어요.',
            '내 삶에 적용': '제가 정말 길들인 관계가 몇이나 되는지 세어보게 됐어요.',
        },
        '사피엔스': {
            '인상깊은 구절 공유': '돈과 국가가 \'상상의 질서\'라는 설명이 충격이었어요.',
            '동의하지 않는 부분': '농업혁명을 인류 최대의 사기라고 단정하는 건 과한 것 같았어요.',
            '내 삶에 적용': '제가 당연하게 여기던 규칙들이 허구일 수 있다는 게 불편하면서 자유로웠어요.',
        },
        '채식주의자': {
            '인상깊은 구절 공유': '영혜가 나무가 되고 싶어 하는 장면이 슬프면서도 이해됐어요.',
            '동의하지 않는 부분': '주변 인물들의 시선으로만 영혜를 그리는 방식이 그녀를 또 대상화하는 것 같았어요.',
            '내 삶에 적용': '저도 묵묵히 따르던 일상의 폭력을 이제야 알아차린 기분이에요.',
        },
        '호밀밭의 파수꾼': {
            '인상깊은 구절 공유': '아이들을 절벽에서 붙잡아 주고 싶다는 홀든의 바람이 뭉클했어요.',
            '동의하지 않는 부분': '홀든이 세상을 위선으로만 보는 태도가 미성숙하게 느껴졌어요.',
            '내 삶에 적용': '저도 순수를 지키려다 정작 아무와도 연결되지 못한 적이 있어요.',
        },
        '자기 앞의 생': {
            '인상깊은 구절 공유': '"사랑할 사람 없이는 살 수 없다"는 모모의 말이 오래 남았어요.',
            '동의하지 않는 부분': '아이의 시점을 빌려 비극을 미화하는 게 아닌가 싶었어요.',
            '내 삶에 적용': '제가 누군가의 자기 앞의 생을 외면해온 건 아닌지 돌아봤어요.',
        },
    }
    
    GOLDEN_SET = [
        {'book_query': b, 'scenario': sc, 'user_msg': msg}
        for b, scs in GOLDEN_BOOKS.items()
        for sc, msg in scs.items()
    ]
    print(f'Golden Set: {len(GOLDEN_SET)} cases (>= {MIN_CASES} 권고 충족: {len(GOLDEN_SET) >= MIN_CASES})')
    assert len(GOLDEN_SET) >= MIN_CASES
print(f"Golden Set: {len(GOLDEN_SET)} cases (>= {MIN_CASES} 권고 충족: {len(GOLDEN_SET) >= MIN_CASES})")
assert len(GOLDEN_SET) >= MIN_CASES

## 2. 각 코칭 스타일별 응답 생성

`USE_API=True`면 실제 백엔드(`/session/start`→`/session/turn`)를 호출한다.
오프라인이면 스타일 특성을 반영한 모의 응답으로 대체해 파이프라인을 재현한다.

In [ ]:
def _mock_assistant(case, style):
    '''스타일 특성을 반영한 모의 코치 응답.
    v2(질문 중심)는 사용자의 말에 닻을 내린 열린 질문을, v1은 평이한 후속 질문을,
    v3은 정중한 반론+도전 질문을 낸다. 일부 케이스에 의도적 환각을 섞어 점검을 시연.'''
    u = case['user_msg'][:24]
    # v1(중립형)에서 일부 케이스에 의도적 환각을 주입해 점검 절차를 시연
    fabricate = (style == 'v1') and (sum(map(ord, case['book_query'])) % 3 == 0)
    if style == 'v2':
        txt = (f'방금 \"{u}…\"라고 하셨는데, 그 표현에서 당신이 전제한 것이 무엇인지 궁금해요. '
               f'그 장면을 조금만 더 묘사해 주실래요? 어떤 지점이 당신을 가장 오래 멈추게 했나요?')
    elif style == 'v3':
        txt = (f'\"{u}…\"라는 입장도 이해되지만, 반대로 이렇게 볼 수도 있지 않을까요? '
               f'당신의 해석이 흔들린다면 어떤 근거 때문일지 되짚어봐 주시겠어요?')
    else:
        txt = (f'좋은 감상이에요. 말씀하신 부분이 인상 깊네요. 이 책을 더 읽고 싶게 만든 점은 무엇인가요?')
    if fabricate:
        txt += ' 특히 3장 마지막 페이지에서 주인공이 \"운명은 없다\"고 외치는 대사가 그렇죠.'  # 요약에 없는 가짜 인용
    return txt, fabricate

def run_one_case_api(case, style):
    r = requests.post(f'{API_URL}/session/start', json={
        'user_pseudo_id': f'eval_{style}', 'book_query': case['book_query'],
        'coaching_style': style}, timeout=60)
    if r.status_code != 200:
        return None
    sid = r.json()['session_id']
    r2 = requests.post(f'{API_URL}/session/turn', json={
        'session_id': sid, 'user_message': case['user_msg']}, timeout=60)
    if r2.status_code != 200:
        return None
    d = r2.json()
    return d['assistant_message'], d.get('thinking_stage'), d['latency_ms'], d['cost_usd']

rows = []
for case in GOLDEN_SET:
    for style in ['v1', 'v2', 'v3']:
        fab_truth = False
        if USE_API:
            res = run_one_case_api(case, style)
            if not res:
                continue
            amsg, stage, lat, cost = res
        else:
            amsg, fab_truth = _mock_assistant(case, style)
            stage = random.choice(['observation','interpretation','evaluation','application'])
            lat = int(random.gauss({'v1':1500,'v2':2200,'v3':2600}[style], 400))
            cost = round(random.gauss(0.0011, 0.0002), 7)
        rows.append({'book': case['book_query'], 'scenario': case['scenario'], 'style': style,
                     'user_msg': case['user_msg'], 'assistant_msg': amsg,
                     'thinking_stage': stage, 'latency_ms': max(lat, 200),
                     'cost_usd': max(cost, 0.0), '_fab_truth': fab_truth})
        if USE_API:
            time.sleep(0.3)

df_gen = pd.DataFrame(rows)
print(len(df_gen), 'responses')
df_gen.head()


## 3. LLM-as-a-Judge 채점 (+ 환각 플래그)

4축 1~5점 + `fabrication`(책 요약/사용자 발화에 없는 장면을 지어냈으면 1).

In [ ]:
JUDGE_RUBRIC = '''당신은 독서 토론 코칭 응답의 "질문 품질"을 평가하는 엄격한 심사위원입니다.
각 축 1~5점으로 채점하세요.
- openness (0.2): Yes/No로 닫히지 않고 사고를 확장시키는 열린 질문인가
- groundedness (0.3): 사용자 발화 또는 제공된 책 메타에 닻을 내렸는가 (지어낸 인용이면 낮게)
- specificity (0.2): 추상적 일반론이 아니라 구체성이 있는가
- provocativeness (0.3): 사용자가 다시 답하고 싶을 만큼 흥미로운가
또한 fabrication(0 또는 1): 책 요약/사용자 발화에 없는 구체적 장면·대사·페이지를 지어냈으면 1.

[책] {book}
[사용자 발화] {user_msg}
[코치 응답 — 평가 대상] {assistant_msg}

JSON만 출력:
{{"openness": int, "groundedness": int, "specificity": int, "provocativeness": int, "fabrication": int, "justification": "한 문장"}}'''

W = {'openness':0.2,'groundedness':0.3,'specificity':0.2,'provocativeness':0.3}

def _depth(d):
    return round(sum(d[k]*w for k,w in W.items()), 3)

def judge_api(row):
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    prompt = JUDGE_RUBRIC.format(book=row['book'], user_msg=row['user_msg'], assistant_msg=row['assistant_msg'])
    resp = client.chat.completions.create(model=JUDGE_MODEL, temperature=0.0, max_tokens=300,
        messages=[{'role':'user','content':prompt}], response_format={'type':'json_object'})
    return json.loads(resp.choices[0].message.content)

def judge_mock(row):
    '''스타일별 품질 프로파일을 반영한 모의 채점 (v2 > v3 > v1 경향).'''
    base = {'v1': 3.0, 'v2': 4.3, 'v3': 3.9}[row['style']]
    def s(mu):
        return int(min(5, max(1, round(random.gauss(mu, 0.5)))))
    d = {'openness': s(base if row['style']!='v1' else base-0.3),
         'groundedness': s(base+0.2), 'specificity': s(base-0.2),
         'provocativeness': s(base if row['style']=='v3' else base-0.1)}
    fab = 1 if row.get('_fab_truth') else 0
    if fab:  # 환각이면 groundedness를 강하게 끌어내림
        d['groundedness'] = min(d['groundedness'], 2)
    d['fabrication'] = fab
    d['justification'] = '모의 채점'
    return d

judgements = []
for _, r in df_gen.iterrows():
    j = judge_api(r) if USE_API else judge_mock(r)
    j.setdefault('fabrication', 0)
    j['depth_score'] = _depth(j)
    for c in ['style','book','scenario']:
        j[c] = r[c]
    judgements.append(j)
    if USE_API:
        time.sleep(0.2)

df_eval = pd.DataFrame(judgements)
print('fabrication flagged:', int(df_eval['fabrication'].sum()), '/', len(df_eval))
df_eval.head()


## 4. 스타일별 집계 — A/B 결정 근거 (비용·지연·안전·품질)

In [ ]:
axes = ['openness','groundedness','specificity','provocativeness','depth_score']
summary = df_eval.groupby('style')[axes].mean().round(2)
summary['fabrication_rate'] = df_eval.groupby('style')['fabrication'].mean().round(3)
summary['latency_ms'] = df_gen.groupby('style')['latency_ms'].mean().round(0)
summary['cost_usd'] = df_gen.groupby('style')['cost_usd'].mean().round(6)
summary


## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
fig, axx = plt.subplots(1, 3, figsize=(15, 4))
summary['depth_score'].plot(kind='bar', ax=axx[0], color=['#888','#1F4E79','#C0504D'])
axx[0].axhline(4.0, color='gray', ls='--', label='Target 4.0'); axx[0].set_ylim(0,5)
axx[0].set_title('Question Depth Score'); axx[0].legend()
summary['fabrication_rate'].plot(kind='bar', ax=axx[1], color=['#888','#1F4E79','#C0504D'])
axx[1].set_title('Fabrication Rate (lower=better)'); axx[1].set_ylim(0, max(0.2, summary['fabrication_rate'].max()*1.3+0.01))
summary[['latency_ms']].plot(kind='bar', ax=axx[2], legend=False, color='#444')
axx[2].set_title('Avg Latency (ms)')
plt.tight_layout(); plt.savefig('eval_result.png', dpi=120); plt.show()


## 6. 사람 평가와의 상관 검증 (Calibration) — 평가 신뢰도의 핵심

LLM Judge를 믿어도 되는지 확인하려면, 골든셋 일부를 **사람이 같은 4축으로 채점**하고
Judge 점수와의 **Spearman 상관**을 본다. 상관이 임계값(`MIN_CORR`) 미만이면 Judge를 재튜닝한다.

- 운영: `human_ratings_template.csv`를 내보내 평가자에게 배포 → 채워진 CSV를 `human_ratings.csv`로 저장.
- 이 노트북: `human_ratings.csv`가 있으면 그것을, 없으면(오프라인 데모) 사람 채점을 모의 생성해 절차를 시연한다.

In [ ]:
# 6-1. 사람 채점용 템플릿 내보내기 (depth_score 기준 상·하위 고루 30건 샘플)
cal = df_eval.sort_values('depth_score').reset_index(drop=True)
idx = list(range(0, len(cal), max(1, len(cal)//MIN_CASES)))[:MIN_CASES]
template = df_gen.loc[df_gen.index.isin([]), :]  # placeholder
cal_sample = df_eval.iloc[idx].copy()
gen_lut = df_gen.set_index(['book','scenario','style'])['assistant_msg']
def _amsg(r):
    try: return gen_lut.loc[(r['book'], r['scenario'], r['style'])]
    except Exception: return ''
tmpl = pd.DataFrame({
    'book': cal_sample['book'], 'scenario': cal_sample['scenario'], 'style': cal_sample['style'],
    'assistant_msg': cal_sample.apply(_amsg, axis=1),
    'human_openness':'', 'human_groundedness':'', 'human_specificity':'', 'human_provocativeness':'',
})
tmpl.to_csv('human_ratings_template.csv', index=False, encoding='utf-8-sig')
print('내보냄: human_ratings_template.csv  (행 수:', len(tmpl), ')')


In [ ]:
# 6-2. 사람 채점 로드 (없으면 모의 생성) → Judge와 Spearman 상관
HUMAN_CSV = 'human_ratings.csv'
HW = {'openness':0.2,'groundedness':0.3,'specificity':0.2,'provocativeness':0.3}

if os.path.exists(HUMAN_CSV):
    human = pd.read_csv(HUMAN_CSV)
    human_source = 'human_ratings.csv (실제 사람 채점)'
else:
    # 오프라인 데모: 사람 채점을 'Judge에 강한 상관 + 약간의 불일치'로 모의
    human = cal_sample[['book','scenario','style','openness','groundedness','specificity','provocativeness']].copy()
    for ax in ['openness','groundedness','specificity','provocativeness']:
        human['human_'+ax] = (human[ax] + pd.Series(
            [random.gauss(0, 0.33) for _ in range(len(human))], index=human.index)
        ).round().clip(1,5)
    human = human[['book','scenario','style','human_openness','human_groundedness','human_specificity','human_provocativeness']]
    human_source = '모의 사람 채점 (오프라인 데모)'

def spearman(a, b):
    '''scipy 없이 Spearman 상관 = 순위(rank)에 대한 Pearson 상관.'''
    a = pd.to_numeric(pd.Series(list(a)), errors='coerce')
    b = pd.to_numeric(pd.Series(list(b)), errors='coerce')
    m = a.notna() & b.notna()
    ra, rb = a[m].rank(), b[m].rank()
    if len(ra) < 2 or ra.std() == 0 or rb.std() == 0:
        return float('nan')
    return float(((ra - ra.mean()) * (rb - rb.mean())).sum() /
                 (math.sqrt(((ra-ra.mean())**2).sum()) * math.sqrt(((rb-rb.mean())**2).sum())))

merged = cal_sample.merge(human, on=['book','scenario','style'], how='inner')
merged['human_depth'] = sum(pd.to_numeric(merged['human_'+a], errors='coerce')*w for a,w in HW.items())

corr_rows = []
for ax in ['openness','groundedness','specificity','provocativeness']:
    corr_rows.append({'axis': ax, 'spearman_rho': round(spearman(merged[ax], merged['human_'+ax]), 3)})
depth_rho = spearman(merged['depth_score'], merged['human_depth'])
corr_rows.append({'axis': 'depth_score', 'spearman_rho': round(depth_rho, 3)})
corr_df = pd.DataFrame(corr_rows)

passed = bool(depth_rho >= MIN_CORR)
print(f'사람 채점 출처: {human_source}  (n={len(merged)})')
print(corr_df.to_string(index=False))
print(f'\nDepth 상관 ρ={depth_rho:.3f}  vs 임계값 {MIN_CORR}  →  {"통과 ✅ Judge 신뢰 가능" if passed else "미달 ❌ Judge 프롬프트 재튜닝 필요"}')


## 7. 대시보드용 산출물 저장 — `eval_summary.json`

`/ops/overview`가 이 파일을 읽어 대시보드 "운영 종합" 패널의 품질 점수로 노출한다.

In [ ]:
eval_summary = {
    'judge_model': JUDGE_MODEL,
    'mode': 'live_api' if USE_API else 'offline_mock',
    'n_golden_cases': len(GOLDEN_SET),
    'by_style': [
        {'style': s,
         'depth_score': float(summary.loc[s, 'depth_score']),
         'groundedness': float(summary.loc[s, 'groundedness']),
         'fabrication_rate': float(summary.loc[s, 'fabrication_rate']),
         'avg_latency_ms': float(summary.loc[s, 'latency_ms']),
         'avg_cost_usd': float(summary.loc[s, 'cost_usd'])}
        for s in summary.index
    ],
    'calibration': {
        'human_source': human_source,
        'n_calibration': int(len(merged)),
        'spearman': {r['axis']: r['spearman_rho'] for r in corr_rows},
        'min_correlation_threshold': MIN_CORR,
        'passed': passed,
    },
}
with open('eval_summary.json', 'w', encoding='utf-8') as f:
    json.dump(eval_summary, f, ensure_ascii=False, indent=2)
print(json.dumps(eval_summary, ensure_ascii=False, indent=2))


## 8. 결론 가이드 (학습 없이 운영하는 회귀 루프)

- **채택 기준**: Depth Score 최고 + `fabrication_rate` 최저 + p95 지연/비용 KPI 충족 스타일을 채택(기본 v2 가설).
- **신뢰 기준**: 위 6번 Calibration의 Depth 상관 ρ ≥ 0.70일 때만 Judge 점수를 의사결정에 사용.
- **회귀 게이트**: 프롬프트(`configs/prompts.yaml`)나 모델을 바꾸면 이 노트북을 재실행해
  Depth Score가 하락하지 않는지, `fabrication_rate`가 오르지 않는지 확인하고 `eval_summary.json`을 커밋한다.
- **개선은 학습이 아니라 프롬프트로**: 점수가 낮은 축은 해당 스타일 프롬프트의 규칙/예시를 고쳐
  같은 골든셋으로 재평가한다. (Reward Model/DPO 없음 — 피드백 반영)
